# Part 1 - Job Shop Scheduling (working notebook)

**Not part of the submission.** This notebook exists to make the scheduling logic visible so you can write the report.

The core of every question is one formula, applied to each action in order:

```
start  = max( operation ready time , machine free time )
finish = start + processing time
```

- **ready time** = the job's own constraints (arrival for a first op; predecessor's finish for a second op)
- **machine free time** = the resource constraint (a machine runs one op at a time)

Whichever is *later* decides when the action actually starts.

Run the cells top to bottom (Shift+Enter).

## Step 1 - Problem data

The 3 jobs / 2 machines table from the assignment, encoded as Python.

In [ ]:
# When each job arrives (its first operation cannot start before this time).
ARRIVAL = {"J1": 0, "J2": 10, "J3": 20}

# op_name : (job, machine, proc_time, is_first_op)
OPS = {
    "O11": ("J1", "M1", 50, True),   # J1's first op
    "O12": ("J1", "M2", 25, False),  # J1's second op (needs O11 done first)
    "O21": ("J2", "M2", 30, True),   # J2's first op
    "O22": ("J2", "M1", 35, False),  # J2's second op (needs O21 done first)
    "O31": ("J3", "M1", 40, True),   # J3's first op
    "O32": ("J3", "M2", 20, False),  # J3's second op (needs O31 done first)
}

# First operation of each job, so a second op can find when its predecessor finished.
FIRST_OP = {"J1": "O11", "J2": "O21", "J3": "O31"}

print("Loaded", len(OPS), "operations across", len(ARRIVAL), "jobs.")

## Step 2 - The scheduler

Walk the action sequence in order and time each action with `start = max(ready, machine free)`.
Returns one result dict per action so we can print the full reasoning afterwards.

In [ ]:
def schedule(sequence):
    """Given an ordered list of operations, compute start/finish for each."""
    machine_free = {"M1": 0, "M2": 0}  # earliest time each machine is idle
    op_finish = {}                     # op_name -> its finish time (precedence)
    results = []

    for op in sequence:
        job, machine, proc, is_first = OPS[op]

        # Step 1: operation ready time
        # First op  -> ready at the job's arrival time.
        # Second op -> ready when the job's first op FINISHED.
        if is_first:
            ready = ARRIVAL[job]
            ready_reason = f"{job} arrives at {ready}"
        else:
            predecessor = FIRST_OP[job]
            ready = op_finish[predecessor]
            ready_reason = f"{predecessor} finishes at {ready} (precedence)"

        # Step 2: machine free time
        free = machine_free[machine]

        # Step 3: start = max(ready, machine free)
        start = max(ready, free)

        # Which constraint actually decided the start time? (useful for the report)
        if ready > free:
            bound = "waiting on the operation (ready time)"
        elif free > ready:
            bound = f"waiting on the machine ({machine} busy until {free})"
        else:
            bound = "both became available at the same moment"

        # Step 4: finish = start + processing time
        finish = start + proc

        # Step 5: update state for later actions
        machine_free[machine] = finish  # this machine is now busy until finish
        op_finish[op] = finish          # remember finish for any successor op

        results.append({
            "op": op, "machine": machine, "proc": proc,
            "ready": ready, "ready_reason": ready_reason,
            "free": free, "start": start, "finish": finish, "bound": bound,
        })

    return results

print("schedule() defined.")

## Step 3 - Pretty-print the reasoning

Prints each action's ready time, machine free time, the `max`, and which constraint won -
plus a summary of t1..t6 and the non-decreasing consistency check.

In [ ]:
def report(sequence):
    results = schedule(sequence)

    print("=" * 70)
    print("Earliest start times")
    print("Sequence:", "  ->  ".join(sequence))
    print("=" * 70)

    for i, r in enumerate(results, start=1):
        print(f"\nAction {i}:  Process({r['op']}, {r['machine']}, t{i})")
        print(f"  ready time   = {r['ready']:>3}   ({r['ready_reason']})")
        print(f"  {r['machine']} free at   = {r['free']:>3}")
        print(f"  start  t{i}    = max({r['ready']}, {r['free']}) = {r['start']:>3}"
              f"   <- {r['bound']}")
        print(f"  finish       = {r['start']} + {r['proc']} = {r['finish']:>3}")

    # Summary line of t1..tN, in the assignment's notation.
    print("\n" + "-" * 70)
    starts = [r["start"] for r in results]
    print("Summary:  " + ",  ".join(
        f"t{i}={s}" for i, s in enumerate(starts, start=1)))

    # The sequence is sorted by start time, so starts must be non-decreasing.
    ok = all(a <= b for a, b in zip(starts, starts[1:]))
    print(f"Non-decreasing check (t1 <= ... <= tN): {'PASS' if ok else 'FAIL'}")
    print("-" * 70)

    return results

print("report() defined.")

## Step 4 - Q1: run the given FCFS sequence

`O11 -> O21 -> O31 -> O12 -> O22 -> O32`

In [ ]:
fcfs_sequence = ["O11", "O21", "O31", "O12", "O22", "O32"]
results = report(fcfs_sequence)